# 05.10 · Choosing and saving the final model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/10-choosing-and-saving/lesson.ipynb)

Read the [README](README.md) first and apply the one-standard-error rule by hand. This notebook:

1. Builds a leaderboard of five churn models: score, stability, speed, interpretability.
2. Chooses one by business need, not just score.
3. Saves the whole pipeline, its threshold and its metadata in one versioned file.
4. Reloads it as if in a new program and scores a brand-new customer.
5. Writes a model card.

## 1. Data and candidates

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

def expected_cost(y, p, t, c_voucher=5, c_lost=60):
    """Voucher for everyone with p >= t; a missed churner costs c_lost."""
    y, flag = np.asarray(y), np.asarray(p) >= t
    return c_voucher * flag.sum() + c_lost * ((y == 1) & ~flag).sum()

from pathlib import Path
def models_dir():
    """<repo>/models when run inside the course repo, ./models otherwise (e.g. Colab)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "download.py").exists():
            d = p / "models"; break
    else:
        d = Path("models")
    d.mkdir(exist_ok=True)
    return d

3,317 customers, churn rate 43.6%, train 2,487 / test 830


In [2]:
import time, lightgbm as lgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(5, shuffle=True, random_state=42)
candidates = {
    "logistic regression": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), "high: one weight per feature"),
    "decision tree (depth 3)": (DecisionTreeClassifier(max_depth=3, random_state=42), "high: 8 readable rules"),
    "KNN (k=51)": (make_pipeline(StandardScaler(), KNeighborsClassifier(51)), "low: 'similar to these customers'"),
    "random forest": (RandomForestClassifier(300, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1), "medium: needs SHAP"),
    "LightGBM (tuned)": (lgb.LGBMClassifier(n_estimators=200, learning_rate=0.01, num_leaves=4,
                                            verbose=-1, random_state=42, n_jobs=1), "medium: needs SHAP"),
}

## 2. The leaderboard

Every number below comes from 5-fold CV on the training customers. The test set is still closed.

In [3]:
rows = []
for name, (m, interp) in candidates.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring="roc_auc", return_train_score=True)
    m.fit(X_train, y_train)
    t = time.perf_counter(); m.predict_proba(X_test); pred_ms = (time.perf_counter() - t) * 1000
    rows.append({"model": name, "CV AUC": r["test_score"].mean(), "CV std": r["test_score"].std(),
                 "train AUC": r["train_score"].mean(), "fit s": r["fit_time"].mean(),
                 "predict ms / 830": pred_ms, "interpretability": interp})
board = pd.DataFrame(rows).set_index("model").sort_values("CV AUC", ascending=False)
board.round(3)

,CV AUC,CV std,train AUC,fit s,predict ms / 830,interpretability
model,,,,,,
LightGBM (tuned),0.744,0.030,0.763,0.010,1.436,medium: needs SHAP
random forest,0.743,0.031,0.801,0.160,26.494,medium: needs SHAP
logistic regression,0.740,0.029,0.744,0.002,0.302,high: one weight per feature
KNN (k=51),0.732,0.033,0.755,0.001,7.069,low: 'similar to these customers'
decision tree (depth 3),0.731,0.024,0.747,0.002,0.221,high: 8 readable rules


How to read it:

- **CV AUC** and **CV std**: score and how much it wobbles between folds.
- **train AUC** vs **CV AUC**: a big gap means the model memorises (overfits).
- **fit s** and **predict ms**: cost to train and to score 830 customers.
- **interpretability**: can you explain a prediction without extra tools?

## 3. The one-standard-error rule

Pick the best CV score; then choose the **simplest** model whose score is within one standard deviation of it.

In [4]:
best = board["CV AUC"].max()
best_std = board.loc[board["CV AUC"].idxmax(), "CV std"]
bar = best - best_std
print(f"best CV AUC {best:.3f}, std {best_std:.3f}  →  anything above {bar:.3f} is a tie")
board["tied with best"] = board["CV AUC"] >= bar
board[["CV AUC", "CV std", "tied with best", "interpretability"]].round(3)

best CV AUC 0.744, std 0.030  →  anything above 0.714 is a tie


,CV AUC,CV std,tied with best,interpretability
model,,,,
LightGBM (tuned),0.744,0.030,True,medium: needs SHAP
random forest,0.743,0.031,True,medium: needs SHAP
logistic regression,0.740,0.029,True,high: one weight per feature
KNN (k=51),0.732,0.033,True,low: 'similar to these customers'
decision tree (depth 3),0.731,0.024,True,high: 8 readable rules


**All five models** are within one standard deviation of the best: on this table, the leaderboard order is mostly noise. The rule says choose the simplest, which leaves two candidates: the depth-3 tree and logistic regression.

Logistic regression wins that pair. Its CV AUC is a little higher, it is the fastest to train and score, its coefficients explain direction to a retention manager ("each extra order lowers the risk"), and it gives a smooth probability for every customer. The depth-3 tree gives only 8 distinct probabilities, which makes threshold choice coarse. **It is the right choice here**, even though it is third in the table.

On a different problem with real non-linear structure, such as the 20,000-order fraud data in 05.4 (logistic 0.815 vs LightGBM 0.874), boosting wins by a margin far larger than the noise, and you would choose it. The rule is the same; the answer depends on the evidence.

## 4. Threshold, then test once

The model outputs a probability; ShopWave needs a yes/no voucher decision. Choose the threshold by expected cost (£5 voucher, £60 lost customer, as in 03.7) on **cross-validated training predictions**.

In [5]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
final = candidates["logistic regression"][0]
p_cv = cross_val_predict(final, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
grid_t = np.round(np.arange(0.05, 0.96, 0.01), 2)
costs = [expected_cost(y_train, p_cv, t) for t in grid_t]
threshold = float(grid_t[int(np.argmin(costs))])
print(f"cheapest threshold on training CV: {threshold}")

final.fit(X_train, y_train)
p_test = final.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, p_test)
print(f"test AUC {test_auc:.3f}   test cost at t={threshold}: £{expected_cost(y_test, p_test, threshold):,}")

cheapest threshold on training CV: 0.07
test AUC 0.741   test cost at t=0.07: £3,945


The theory says the same thing. Send a voucher when the expected loss avoided is bigger than the voucher's cost: $p 	imes £60 > £5$, so $p > 5/60 = 0.083$. Training CV picked 0.07, close to that, because logistic regression's probabilities are well calibrated.

At these prices a voucher is so cheap compared with a lost customer that about 94% of test customers get one. The model's value is in confidently skipping the other 6%: £3,945 against £4,150 for "voucher to everyone". With a £20 voucher the threshold would rise to about 0.33 and the model would matter much more (exercise idea).

## 5. Save everything the model needs, in one file

A saved model is useless without the exact feature list, the threshold and a record of how good it was. Save them together, with the date and the score in the file name so versions never overwrite each other.

In [6]:
import joblib, datetime, sklearn
today = datetime.date.today().isoformat()
bundle = {
    "pipeline": final,                       # scaler + logistic regression, already fitted
    "features": features,                    # exact column order the pipeline expects
    "threshold": threshold,
    "metrics": {"cv_auc": float(board.loc["logistic regression", "CV AUC"]), "test_auc": float(test_auc)},
    "trained_on": f"Online Retail II, orders before {cutoff.date()}", "trained": today,
    "sklearn_version": sklearn.__version__,
}
path = models_dir() / f"churn_logreg_{today}_auc{test_auc:.3f}.joblib"
joblib.dump(bundle, path)
print("saved:", path.name, f"({path.stat().st_size / 1024:.1f} KB)")

saved: churn_logreg_2026-10-28_auc0.741.joblib (2.0 KB)


## 6. Reload and score a new customer

Imagine this is a different program, a week later. All it has is the file and one customer's raw numbers.

In [7]:
loaded = joblib.load(path)
new_customer = {"orders": 2, "spend": 310.50, "products": 14, "tenure_days": 150, "days_since_last": 95}

row = pd.DataFrame([new_customer])[loaded["features"]]          # same columns, same order
p = loaded["pipeline"].predict_proba(row)[0, 1]
decision = "send £5 voucher" if p >= loaded["threshold"] else "no voucher"
print(f"P(churn) = {p:.2f}  →  {decision}   (threshold {loaded['threshold']})")

P(churn) = 0.52  →  send £5 voucher   (threshold 0.07)


In [8]:
import warnings
if loaded["sklearn_version"] != sklearn.__version__:
    warnings.warn(f"model saved with scikit-learn {loaded['sklearn_version']}, running {sklearn.__version__}")
print("version check done:", loaded["sklearn_version"])

version check done: 1.9.1


## 7. The model card

A model card is a one-page description that travels with the model: what it is for, what it was trained on, how good it is, and where it should not be used. Here it is generated from the bundle so the numbers are always right.

In [9]:
from IPython.display import Markdown
coefs = pd.Series(final[-1].coef_[0], index=features).sort_values()
card = f"""
### Model card: ShopWave churn model ({loaded['trained']})

| | |
|---|---|
| **Purpose** | Rank customers by risk of not ordering in the next 90 days; send a £5 voucher above the threshold |
| **Model** | Standardised logistic regression, scikit-learn {loaded['sklearn_version']} |
| **Training data** | {loaded['trained_on']}; {len(X_train):,} customers (75% of {len(cust):,}) |
| **Inputs** | {", ".join(features)} |
| **Performance** | CV AUC {loaded['metrics']['cv_auc']:.3f}; test AUC {loaded['metrics']['test_auc']:.3f} |
| **Decision rule** | Voucher if P(churn) ≥ {loaded['threshold']} (cheapest at £5 voucher / £60 lost customer) |
| **Strongest signals** | fewer orders → higher risk ({coefs.idxmin()}: {coefs.min():+.2f}); {coefs.idxmax()} ({coefs.max():+.2f}) |
| **Not for** | New customers with no order history; wholesale accounts; any decision other than vouchers |
| **Known limits** | Trained on one autumn; seasonal behaviour may differ. Retrain at least quarterly. |
"""
Markdown(card)


### Model card: ShopWave churn model (2026-10-28)

| | |
|---|---|
| **Purpose** | Rank customers by risk of not ordering in the next 90 days; send a £5 voucher above the threshold |
| **Model** | Standardised logistic regression, scikit-learn 1.9.1 |
| **Training data** | Online Retail II, orders before 2011-09-01; 2,487 customers (75% of 3,317) |
| **Inputs** | orders, spend, products, tenure_days, days_since_last |
| **Performance** | CV AUC 0.740; test AUC 0.741 |
| **Decision rule** | Voucher if P(churn) ≥ 0.07 (cheapest at £5 voucher / £60 lost customer) |
| **Strongest signals** | fewer orders → higher risk (orders: -0.99); days_since_last (+0.44) |
| **Not for** | New customers with no order history; wholesale accounts; any decision other than vouchers |
| **Known limits** | Trained on one autumn; seasonal behaviour may differ. Retrain at least quarterly. |


## Quiz answers

1. Score alone ignores stability, cost, speed, interpretability and robustness. A model that is 0.002 better but impossible to explain or 100× slower is usually the wrong choice.
2. Best 0.750 with std 0.020 → the bar is 0.730. Models at 0.745 and 0.735 are tied with the best; 0.720 is not. Choose the simplest of the tied ones.
3. The fitted pipeline (including preprocessing), the exact feature list in order, the decision threshold, its metrics, the training data description and date, and library versions.
4. So that a new version never overwrites an old one, and anyone can see at a glance which file is which and how good it was.
5. Purpose, model type, training data, inputs, performance, decision rule, limitations and where not to use it.

**Next:** [05.P Mini-project: a tuned, explained churn model](../project/)